In [58]:
import jax
import os

jax_cache_dir = os.path.join(os.path.dirname(os.path.dirname(__vsc_ipynb_file__)), ".jax_cache")

jax.config.update("jax_compilation_cache_dir", jax_cache_dir)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

print('jax cache dir:', jax_cache_dir)

jax cache dir: /home/mat/ml/flax-qwen3/.jax_cache


In [59]:
from huggingface_hub import snapshot_download
from transformers import PreTrainedTokenizerFast

model_id = "Qwen/Qwen3-0.6B-Base"

snapshot_dir = snapshot_download(model_id)

tokenizer = PreTrainedTokenizerFast.from_pretrained(model_id)

print('snapshot_dl:', snapshot_dir)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

snapshot_dl: /home/mat/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/da87bfb608c14b7cf20ba1ce41287e8de496c0cd


In [60]:
import json
from pathlib import Path
from safetensors import safe_open

cfg = json.loads((Path(snapshot_dir) / "config.json").read_text())
L, N, K, H, D = cfg['num_hidden_layers'], cfg['num_attention_heads'], cfg['num_key_value_heads'], cfg['head_dim'], cfg['hidden_size']

weights = dict()
with safe_open(Path(snapshot_dir) / "model.safetensors", framework="flax") as f:
    for key in f.keys():
        weights[key] = f.get_tensor(key)


In [61]:
weights.keys()


dict_keys(['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_norm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.1.self_attn.k_norm.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.self_attn.q_norm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.10.input_layernorm.weight', 'm

In [62]:
weights['model.layers.0.mlp.up_proj.weight'].mean()

Array(-1.50204e-05, dtype=bfloat16)

In [ ]:
import flax.linen as nn
import jax.numpy as j

class Qwen3Model(nn.Module):
  vocab_size: int
  hidden_size: int
  rms_norm_eps: float
  num_hidden_layers: int
  num_attention_heads: int
  num_key_value_heads: int
  head_dim: int

  @nn.compact
  def __call__(self, x: jax.Array):
    x = nn.Embed(self.vocab_size, self.hidden_size)(x)

    for i in range(self.num_hidden_layers):
      x = nn.RMSNorm(self.rms_norm_eps)(x)
      
      q = nn.Dense(self.num_attention_heads * self.head_dim, use_bias=False, name=f"q_proj_{i}")(x)
      q = q.reshape(-1, self.num_attention_heads, self.head_dim)

      k = nn.Dense(self.num_key_value_heads * self.head_dim, use_bias=False, name=f"k_proj_{i}")(x)
      k = k.reshape(-1, self.num_key_value_heads, self.head_dim)

      v = nn.Dense(self.num_key_value_heads * self.head_dim, use_bias=False, name=f"v_proj_{i}")(x)
      v = v.reshape(-1, self.num_key_value_heads, self.head_dim)

    return x


In [133]:
# init model

key = jax.random.PRNGKey(0)
model = Qwen3Model(
  vocab_size=cfg["vocab_size"],
  hidden_size=cfg["hidden_size"],
  rms_norm_eps=cfg["rms_norm_eps"],
  num_hidden_layers=cfg["num_hidden_layers"],
  num_attention_heads=cfg['num_attention_heads'],
  num_key_value_heads=cfg['num_key_value_heads'],
  head_dim=cfg['head_dim']
)

# params: convert the names from the Qwen3 snapshot to the Flax names
state = {
  'params': {
    'Embed_0': {
      'embedding': weights['model.embed_tokens.weight']
    }
  }
}

for i in range(cfg["num_hidden_layers"]):
  state['params'][f'RMSNorm_{i}'] = {
    'scale': weights[f'model.layers.{i}.input_layernorm.weight']
  }
  state['params'][f'q_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.q_proj.weight'].transpose(1, 0),
  }
  state['params'][f'k_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.k_proj.weight'].transpose(1, 0),
  }
  state['params'][f'v_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.v_proj.weight'].transpose(1, 0),
  }

# state = model.init(key, j.ones((32), dtype=j.int32))
# params = state['params']
# params['Embed_0']['embedding'] = weights['model.embed_tokens.weight']


In [124]:
print('Flax weights: ', jax.tree_util.tree_structure(state))
print('Original weights: ', jax.tree_util.tree_structure(weights))

Flax weights:  PyTreeDef({'params': {'Embed_0': {'embedding': *}, 'RMSNorm_0': {'scale': *}, 'RMSNorm_1': {'scale': *}, 'RMSNorm_10': {'scale': *}, 'RMSNorm_11': {'scale': *}, 'RMSNorm_12': {'scale': *}, 'RMSNorm_13': {'scale': *}, 'RMSNorm_14': {'scale': *}, 'RMSNorm_15': {'scale': *}, 'RMSNorm_16': {'scale': *}, 'RMSNorm_17': {'scale': *}, 'RMSNorm_18': {'scale': *}, 'RMSNorm_19': {'scale': *}, 'RMSNorm_2': {'scale': *}, 'RMSNorm_20': {'scale': *}, 'RMSNorm_21': {'scale': *}, 'RMSNorm_22': {'scale': *}, 'RMSNorm_23': {'scale': *}, 'RMSNorm_24': {'scale': *}, 'RMSNorm_25': {'scale': *}, 'RMSNorm_26': {'scale': *}, 'RMSNorm_27': {'scale': *}, 'RMSNorm_3': {'scale': *}, 'RMSNorm_4': {'scale': *}, 'RMSNorm_5': {'scale': *}, 'RMSNorm_6': {'scale': *}, 'RMSNorm_7': {'scale': *}, 'RMSNorm_8': {'scale': *}, 'RMSNorm_9': {'scale': *}, 'k_proj_0': {'kernel': *}, 'k_proj_1': {'kernel': *}, 'k_proj_10': {'kernel': *}, 'k_proj_11': {'kernel': *}, 'k_proj_12': {'kernel': *}, 'k_proj_13': {'kernel'

In [127]:
weights[f'model.layers.{i}.self_attn.q_proj.weight'].shape

(2048, 1024)

In [134]:
# forward pass (random input)
x = jax.random.randint(key, (32), 0, cfg['vocab_size'])
out = model.apply(state, x)
print(out.shape)
print(out)


(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
(32, 16, 128) (32, 8, 128) (32, 8, 128)
